In [ ]:
import pandas as pd
import numpy as np
import pickle
import os

PATH = '/data/'
seed = 2025
dataset_name = 'amazon-game'
data_path = os.path.join(PATH, f'{dataset_name}/')
DATASET_PATH = os.path.join(data_path, f'original_data')
if not os.path.exists(data_path):
    os.makedirs(data_path)
saved_processed_data_path = os.path.join(data_path, f'processed_data')
if not os.path.exists(saved_processed_data_path):
    os.makedirs(saved_processed_data_path)

In [ ]:
interactions = pd.read_csv(os.path.join(DATASET_PATH, "amazon_Video_Games.csv.gz"))
item_info = pd.read_csv(os.path.join(DATASET_PATH, "amazon_Video_Games_info.csv.gz"))
print(interactions.head())
print(item_info.head())

      item_id                       user_id      timestamp
0  B08R5B7YS4  AEVPPTMG43C6GWSR7I2UGRQN7WFQ  1611459666223
1  B0863MT183  AEVPPTMG43C6GWSR7I2UGRQN7WFQ  1613701986538
2  B08P8P7686  AEVPPTMG43C6GWSR7I2UGRQN7WFQ  1613702112995
3  B0B7LV3DN2  AEVPPTMG43C6GWSR7I2UGRQN7WFQ  1617641445475
4  B09WMQ6DXG  AEVPPTMG43C6GWSR7I2UGRQN7WFQ  1620231368468
      item_id                                        description  \
0  B00Z9TLVK0  Following the record-breaking launch of NBA 2K...   
1  B07H93H878                                                NaN   
2  B00006969T  The Konami Collector's Series: Castlevania & C...   
3  B07DL353F8  A power struggle begins in a Civilization depe...   
4  B00BJH85SW  Product Description\nTurbo: Super Stunt Squad ...   

                                               title  
0   NBA 2K17 - Early Tip Off Edition - PlayStation 4  
1  eXtremeRate Soft Touch Top Shell Front Housing...  
2  Konami Collector's Series: Castlevania & Contr...  
3  Tales of Vespe

In [ ]:
import datetime

def timestamp_to_ddyymm(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime('%d%m%y')

interactions['datetime'] = interactions['timestamp'].apply(timestamp_to_ddyymm)
interactions

,item_id,user_id,timestamp,datetime
0,B08R5B7YS4,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,1611459666223,240121
1,B0863MT183,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,1613701986538,190221
2,B08P8P7686,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,1613702112995,190221
3,B0B7LV3DN2,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,1617641445475,050421
4,B09WMQ6DXG,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,1620231368468,050521
...,...,...,...,...
255284,B0C5K4M7WJ,AGC7POIE3PZI4BOJ43KZE5YFCICA,1522708204570,020418
255285,B0B6HQJM37,AGC7POIE3PZI4BOJ43KZE5YFCICA,1630301922358,300821
255286,B0B75258YP,AGC7POIE3PZI4BOJ43KZE5YFCICA,1685204391685,270523
255287,B07ZTD64SC,AGC7POIE3PZI4BOJ43KZE5YFCICA,1685204683273,270523


In [ ]:
import os
import json

unique_item_ids = interactions['item_id'].unique()

itemid2idx = {str(bid): int(idx) for idx, bid in enumerate(unique_item_ids)}
idx2itemid = {int(idx): str(bid) for bid, idx in itemid2idx.items()}

with open(os.path.join(saved_processed_data_path, 'itemid2idx.json'), 'w', encoding='utf-8') as f:
    json.dump(itemid2idx, f, ensure_ascii=False)

with open(os.path.join(saved_processed_data_path, 'idx2itemid.json'), 'w', encoding='utf-8') as f:
    json.dump(idx2itemid, f, ensure_ascii=False)

interactions['item_id_old'] = interactions['item_id']
interactions['item_id'] = interactions['item_id'].astype(str).map(itemid2idx)

item_info['item_id_old'] = item_info['item_id']
item_info['item_id'] = item_info['item_id'].astype(str).map(itemid2idx)


In [ ]:
sorted_df = interactions.sort_values(by=['user_id', 'datetime'])
grouped = sorted_df.groupby(['user_id', 'datetime'])
list_of_lists = [group['item_id'].tolist() for _, group in grouped]
filtered_lists = [inner_list for inner_list in list_of_lists if 3 < len(inner_list)]

In [9]:
import random
all_item_ids_set = set([item_id for inner_list in filtered_lists for item_id in inner_list])

items_except_last = []
last_item_ids = []
shuffled_item_ids_with_last = []

max_history_length = 25
for inner_list in filtered_lists:
    except_last = inner_list[-max_history_length:-1]
    items_except_last.append(except_last)
    
    last_item_id = inner_list[-1]
    last_item_ids.append(last_item_id)
    
    possible_ids = list(all_item_ids_set.difference(except_last))
    random_ids = random.sample(possible_ids, 49)
    
    random_ids.append(last_item_id)
    random.shuffle(random_ids)
    shuffled_item_ids_with_last.append(random_ids)

In [ ]:
from sklearn.model_selection import train_test_split

test_size = 0.2
zipped_lists = list(zip(items_except_last, last_item_ids, shuffled_item_ids_with_last))
train_zipped, test_zipped = train_test_split(zipped_lists, test_size=test_size, random_state=seed)


if not os.path.exists(saved_processed_data_path):
    os.makedirs(saved_processed_data_path)

with open(os.path.join(saved_processed_data_path, f'{dataset_name}_train.txt'), 'wb') as file:  # Further split train into train and val when training!
    pickle.dump(train_zipped, file)
with open(os.path.join(saved_processed_data_path, f'{dataset_name}_test.txt'), 'wb') as file:
    pickle.dump(test_zipped, file)

In [ ]:
item_info.to_csv(os.path.join(saved_processed_data_path, 'item_info.csv'), index=False)
item_info

,item_id,description,title,item_id_old
0,5272,Following the record-breaking launch of NBA 2K...,NBA 2K17 - Early Tip Off Edition - PlayStation 4,B00Z9TLVK0
1,4721,NaN,eXtremeRate Soft Touch Top Shell Front Housing...,B07H93H878
2,19131,The Konami Collector's Series: Castlevania & C...,Konami Collector's Series: Castlevania & Contr...,B00006969T
3,4330,A power struggle begins in a Civilization depe...,Tales of Vesperia - Definitive Edition - PlayS...,B07DL353F8
4,17275,Product Description\nTurbo: Super Stunt Squad ...,Turbo: Super Stunt Squad - Nintendo 3DS,B00BJH85SW
...,...,...,...,...
20366,19856,NaN,"Switch Controllers,Switch Pro Controller Compa...",B0CHRQXBWC
20367,7198,NaN,KIWI design Ultra-Soft Clip-on Headphones Comp...,B0CHW19XX3
20368,12733,NaN,The Chronicles of Riddick: Escape From Butcher...,B001EYUNNK
20369,19801,An epic adventure across the land and skies of...,The Legend of Zelda: Tears of the Kingdom - Ni...,B0BX7NC5K1
